# 02 — Build and Train a CNN on CIFAR-10
## Main coding practical

We will build a CNN explicitly with `tf.keras.layers`, inspect every output shape and parameter count, train it, and evaluate it. The goal is to understand the operations, not chase a specific accuracy.

## 1. Imports and reproducibility

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

tf.keras.utils.set_random_seed(42)
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

## 2. Load data and inspect dimensions

In [ ]:
(x_train_full, y_train_full), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
class_names = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]
print("Train images:", x_train_full.shape)
print("Train labels:", y_train_full.shape)
print("Test images :", x_test.shape)
print("Test labels :", y_test.shape)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train_full[i])
    ax.set_title(class_names[int(y_train_full[i, 0])])
    ax.axis("off")
plt.tight_layout()

## 3. Normalize and split BEFORE model training

We reserve 20% of the original training set for validation. The official test set remains untouched until final evaluation.

In [ ]:
x_train_full = x_train_full.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Reproducible simple split for teaching; shuffle/group-aware splitting
# may be more appropriate for other datasets.
x_train, y_train = x_train_full[:40000], y_train_full[:40000]
x_val, y_val = x_train_full[40000:], y_train_full[40000:]

print("Training:", x_train.shape, y_train.shape)
print("Validation:", x_val.shape, y_val.shape)
print("Held-out test:", x_test.shape, y_test.shape)

## 4. Build the CNN layer by layer

`Conv2D` learns local filters. `ReLU` keeps positive activations and sets negative values to zero. `MaxPooling2D` selects local maxima. `Flatten` reshapes feature maps into a vector. `Dense` layers learn combinations of features. `Softmax` produces class probabilities.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(32, 32, 3)),
    tf.keras.layers.Conv2D(32, (3, 3), padding="valid", activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), padding="valid", activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax")
])
model.summary()

### Predict the shapes before reading the summary

`32×32×3 → 30×30×32 → 15×15×32 → 13×13×64 → 6×6×64 → 2304 → 64 → 10`

Why does a valid 3×3 stride-1 convolution reduce each spatial dimension by 2? Why does 2×2 pooling with stride 2 halve it?

Parameter examples:
- First Conv2D: `(3×3×3 + 1)×32 = 896`
- Second Conv2D: `(3×3×32 + 1)×64 = 18,496`
- Dense: `2304×64 + 64 = 147,520`

## 5. Compile

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

## 6. Train — watch the training and validation metrics

In [ ]:
EPOCHS = 5
BATCH_SIZE = 64

history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE
)

## 7. Plot learning curves

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["accuracy"], label="Training")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend(); plt.grid(True); plt.show()

plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"], label="Training")
plt.plot(history.history["val_loss"], label="Validation")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.grid(True); plt.show()

## 8. Evaluate on the untouched test set

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

## 9. Save the trained model for the next notebook

In [ ]:
model.save("cifar10_cnn.keras")
print("Saved cifar10_cnn.keras")

## Experiments
Change one item at a time: filter count, batch size, learning rate or epochs. If architecture changes, rebuild and retrain. Do not use the test set to choose hyperparameters.